In [ ]:
import random
import nltk
from nltk import find
import torchaudio
import torch
from torch import nn
from transformers import AutoConfig, AutoModel
import os
from gensim.models import Word2Vec, KeyedVectors
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print('Device available is', device)

seed = 7 
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


In [ ]:
# function to generate audio embeddings

MODEL_NAME = "facebook/wav2vec2-base"
encoder_config = AutoConfig.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME, device_map="cuda")

def generate_embedding(filename, beginning, end):
  # make sure we're only generating an embedding for the specific word
  clip_start = int(beginning * 16000)
  clip_end = int(end * 16000)

  num_frames = clip_end - clip_start

  try:
    waveform,sample_rate=torchaudio.load(filename, frame_offset=clip_start, num_frames=num_frames)
    #ensures both input and encoder are on the same device
    waveform = waveform.to(device)
    # Stereo to mono
    if waveform.shape[0] > 1:
      waveform = torch.mean(waveform,dim=0,keepdim=True)
    # Resample
    if sample_rate != 16000:
      waveform = torchaudio.functional.sample(
          waveform,
          sample_rate,
          16000,
      )
    # extract features
    emission = encoder(waveform.to(device)).last_hidden_state
    emission = emission.detach().cpu()
    # emission = emission.detach().cpu().numpy()
    return emission
  except Exception as e:
    print(f"Error processing {filename}: {e}")
    return None

In [ ]:
# load the transcript data and tokenize sentences
data_path = "LibriSpeech/dev-clean/"

sentences = []
vocab_embeddings = {}
audio_paths = [] 


# load sentences
# for reader_id in os.listdir(data_path):
for reader_idx in range(1):
    reader_id = os.listdir(data_path)[reader_idx]
# for reader_id in tqdm(os.listdir(data_path), desc="Reader", total=len(os.listdir(data_path))):
    reader_path = data_path + str(reader_id) + "/"

    # for chapter_id in os.listdir(reader_path):
    for chapter_id in tqdm(os.listdir(reader_path), desc="Chapter ", total=len(os.listdir(reader_path))):
        chapter_file_path = reader_path + str(chapter_id) + "/"
        transcript_file = str(reader_id) + "-" + str(chapter_id) + ".alignment.txt"

        with open(chapter_file_path + transcript_file) as file:
            for line in file.readlines():
            # for line in tqdm(file.readlines(), desc="Generating for " + chapter_file_path + transcript_file, total=len(file.readlines())):
                # PER SENTENCE LEVEL HERE !
                audio_filename, words, timings = line.split()

                audio_path = chapter_file_path + audio_filename + ".flac"

                words = words.split(",")[1:-1] # first and last entries are always silences, so we remove these
                words = [word.lower() for word in words]
                
                timings = timings.replace('"', "") # remove double quotes
                timings = timings.split(",")
                timings = [float(timing) for timing in timings] # surely there is a better way to do this, but this works for now

                for i in range(len(words)):

                    if words[i] not in vocab_embeddings.keys():
                        vocab_embeddings[words[i]] = []
                    
                    # i+1 and i+2 since the first timing is the end of the silence, and the last one is the ending of the clip-ending silence
                    vocab_embeddings[words[i]].append(generate_embedding(audio_path, timings[i + 1], timings[i + 2]))
                
                sentences.append(words)
            
        # for path in os.listdir(chapter_file_path):
        #     if path.endswith(".flac"):
        #         audio_paths.append(path)

# print(audio_paths)


In [ ]:
# build Word2Vec models
librispeech_model = Word2Vec(sentences)
# librispeech_model.build_vocab(sentences)
print("librispeech Word2Vec model trained from " + str(len(sentences)) + " sentences")

# word2vec model
nltk.download('word2vec_sample')
word2vec_sample = str(find('models/word2vec_sample/pruned.word2vec.txt'))
word2vec_model = KeyedVectors.load_word2vec_format(word2vec_sample, binary=False)

In [ ]:
# text embedding comparisons 
# for key in vocab_embeddings.keys():
#     print(key ,vocab_embeddings[key][0])

libri_word2vec_keys = set(librispeech_model.wv.index_to_key)
word2vec_keys = set(word2vec_model.index_to_key)
wav2vec_keys = set(vocab_embeddings.keys())

both_text_keys = libri_word2vec_keys.intersection(word2vec_keys) # needed because some tokens are not in both sets

# print("Number of word2vec keys: ", len(word2vec_keys))
# print("Number of wav2vec keys: ", len(wav2vec_keys))

test_words = [random.choice(list(both_text_keys)) for i in range(5)]

for word1 in test_words:
    for word2 in test_words:
        # print(word1, word2)
        libriSim = librispeech_model.wv.similarity(word1, word2)
        word2vec_sim = word2vec_model.similarity(word1, word2)

        print("word2vec similarity for " + word1 + " and " + word2 + "\t : " + str(word2vec_sim))
        print("Libri similarity for " + word1 + " and " + word2 + "\t : " + str(libriSim))

In [ ]:
# audio embedding comparisons
cos = nn.CosineSimilarity(dim=0)

for key in wav2vec_keys:
    # print("Embeddings for: " + str(key))
    # print(vocab_embeddings[key][0].size())
    word_embedding_set = vocab_embeddings[key]
    for i in range(len(word_embedding_set) - 1):
        
        vector1 = word_embedding_set[i]
        vector2 = word_embedding_set[i + 1]

        if vector1 is not None and vector2 is not None:
            vector1 = torch.mean(vector1, dim=1)
            vector2 = torch.mean(vector2, dim=1)

            print("vector 1 size: " + str(vector1.size()))
            print("vector 2 size: " + str(vector2.size()))

            print(cos(vector1, vector2))
    



In [ ]:
# check to see the cosine similarity of 